In [11]:
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf

ModuleNotFoundError: No module named 'tensorflow.python'

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
data_dir = os.getenv("DATA_DIR")
datasplits_dir = os.getenv("DATASPLITS_DIR")
embeddings_dir = os.getenv("EMBEDDINGS_DIR")

text_train_path = os.path.join(datasplits_dir, "text_train.tsv")
text_val_path = os.path.join(datasplits_dir, "text_val.tsv")
text_test_path = os.path.join(datasplits_dir, "text_test.tsv")

train_bert_path = os.path.join(embeddings_dir, "train_bert.npy")
val_bert_path = os.path.join(embeddings_dir, "val_bert.npy")
test_bert_path = os.path.join(embeddings_dir, "test_bert.npy")

train_roberta_path = os.path.join(embeddings_dir, "train_roberta.npy")
val_roberta_path = os.path.join(embeddings_dir, "val_roberta.npy")
test_roberta_path = os.path.join(embeddings_dir, "test_roberta.npy")

train_bertweet_path = os.path.join(embeddings_dir, "train_bertweet.npy")
val_bertweet_path = os.path.join(embeddings_dir, "val_bertweet.npy")
test_bertweet_path = os.path.join(embeddings_dir, "test_bertweet.npy")

In [7]:

text_train_df = pd.read_csv(text_train_path, sep='\t')
text_val_df = pd.read_csv(text_val_path, sep='\t')
text_test_df = pd.read_csv(text_test_path, sep='\t')

print(f"text train set size: {len(text_train_df)}")
print(f"text val set size: {len(text_val_df)}")
print(f"text test set size: {len(text_test_df)}")

text_train_df.head()

text train set size: 11240
text val set size: 2409
text test set size: 2409


,tweet_text,text_info,text_human,cleaned_text_bert,cleaned_text_bertweet
0,Second earthquake strikes Mexico in less than ...,informative,other_relevant_information,Second earthquake strikes Mexico in less than ...,Second earthquake strikes Mexico in less than ...
1,Mexico rescinds Harvey assist supply after pur...,informative,rescue_volunteering_or_donation_effort,Mexico rescinds Harvey assist supply after pur...,Mexico rescinds Harvey assist supply after pur...
2,Hurricane Irma Strengthens To Category 5 Targe...,informative,other_relevant_information,Hurricane Irma Strengthens To Category 5 Targe...,Hurricane Irma Strengthens To Category 5 Targe...
3,"SOLIDARITY | After hurricane Maria, a Caribbea...",informative,rescue_volunteering_or_donation_effort,"SOLIDARITY | After hurricane Maria, a Caribbea...","SOLIDARITY | After hurricane Maria, a Caribbea..."
4,Hurricane Irma Nursing home tragedy unfolded d...,informative,other_relevant_information,Hurricane Irma Nursing home tragedy unfolded d...,Hurricane Irma Nursing home tragedy unfolded d...


In [ ]:
# 📌 Task 1: Binary (text_info)
le_info = LabelEncoder()
le_info.fit(text_train_df["text_info"])  # Adatta solo sul train

train_labels_task1 = le_info.transform(text_train_df["text_info"])
val_labels_task1 = le_info.transform(text_val_df["text_info"])
test_labels_task1 = le_info.transform(text_test_df["text_info"])

# 📌 Task 2: Multiclasse (text_human)
le_human = LabelEncoder()
le_human.fit(text_train_df["text_human"])

train_labels_task2 = le_human.transform(text_train_df["text_human"])
val_labels_task2 = le_human.transform(text_val_df["text_human"])
test_labels_task2 = le_human.transform(text_test_df["text_human"])

#Mapping classi e label
print("Task 1 classes:", dict(zip(le_info.classes_, le_info.transform(le_info.classes_))))
print("Task 2 classes:", dict(zip(le_human.classes_, le_human.transform(le_human.classes_))))

Task 1 classes: {'informative': np.int64(0), 'not_informative': np.int64(1)}
Task 2 classes: {'affected_individuals': np.int64(0), 'infrastructure_and_utility_damage': np.int64(1), 'injured_or_dead_people': np.int64(2), 'missing_or_found_people': np.int64(3), 'not_humanitarian': np.int64(4), 'other_relevant_information': np.int64(5), 'rescue_volunteering_or_donation_effort': np.int64(6), 'vehicle_damage': np.int64(7)}


In [ ]:
def create_mlp(input_dim, output_dim, is_binary):
    
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(output_dim, activation='sigmoid' if is_binary else 'softmax')
    ])
    model.compile(
        loss='binary_crossentropy' if is_binary else 'categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    return model

In [ ]:
def plot_training_history(history, title="Model Training"):
    """
    Plot accuracy and loss curves for training and validation.

    Parameters:
        history (History): Keras History object returned by model.fit()
        title (str): Title prefix for the plots
    """
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
def train_and_evaluate(name, X_train, y_train, X_val, y_val, X_test, y_test, task=1):
    is_binary = (task == 1)
    y_train_enc = y_train if is_binary else to_categorical(y_train)
    y_val_enc = y_val if is_binary else to_categorical(y_val)
    
    model = create_mlp(X_train.shape[1], 1 if is_binary else y_train_enc.shape[1], is_binary)
    print(f"\n🚀 Training MLP on {name.upper()} embeddings (Task {task})")

    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = model.fit(
        X_train, y_train_enc,
        validation_data=(X_val, y_val_enc),
        epochs=50,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )

    plot_training_history(history, title=f"{name.upper()} - Task {task}")
    
    y_pred = model.predict(X_test)
    y_pred_labels = (y_pred > 0.5).astype(int).flatten() if is_binary else np.argmax(y_pred, axis=1)

    print(f"\n📊 Results for {name.upper()} - Task {task}")
    print("Accuracy:", accuracy_score(y_test, y_pred_labels))
    print(classification_report(y_test, y_pred_labels, zero_division=0))

In [ ]:
# 🚀 Loop su ogni embedding
for name, paths in embedding_sets.items():
    X_train = np.load(paths["train"])
    X_val = np.load(paths["val"])
    X_test = np.load(paths["test"])

    # 🧪 Task 1: Informative vs Not informative (binario)
    train_and_evaluate(name, X_train, train_labels_task1, X_val, val_labels_task1, X_test, test_labels_task1, task=1)

    # 🧪 Task 2: Humanitarian categories (multiclasse)
    train_and_evaluate(name, X_train, train_labels_task2, X_val, val_labels_task2, X_test, test_labels_task2, task=2)